# 03 · 参数扫描

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next((candidate for candidate in (start, *start.parents) if (candidate / 'vbt').exists()), start)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'项目根目录: {ROOT}')

参数扫描走纯矩阵研究路径，可多核并行；最佳参数仍需在完整规则/RQAlpha 中复核。

## 1. 数据与参数网格

In [ ]:
from vbt.config import load_backtest_config, load_scan_config, load_strategy_config
from vbt.adapters import DEFAULT_FACTORS, VBTDataLoader
config = load_backtest_config()
scan_config = load_scan_config()
params = load_strategy_config({'alignment_mode': False})
data = VBTDataLoader(start_date=config['start_date'], end_date=config['end_date']).load(factors=DEFAULT_FACTORS)
scan_config

## 2. 并行运行

In [ ]:
from vbt.engine import VBTEngine
from vbt.engine.parameter_scan import ParameterScan
from vbt.strategies import DividendLowVolStrategy
engine = VBTEngine(data=data, strategy=DividendLowVolStrategy(params), initial_capital=config['initial_capital'], backtest_config=config)
scan_results = ParameterScan(engine=engine, param_grid=scan_config['param_grid'], metric=scan_config['metric']).run(scan_config['n_jobs'])
display(scan_results.table.head(scan_config['top_k']))
scan_results.best_params()

## 3. 热力图与平行坐标

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
table = scan_results.table.query("status == 'ok'")
annual = table[table['rebalance_freq'].eq('A')]
if not annual.empty:
    sns.heatmap(annual.pivot(index='top_n', columns='volatility_60d_max', values=scan_config['metric']), annot=True, fmt='.2f')
    plt.title('年度调仓参数热力图')
import plotly.express as px
import plotly.io as pio
from IPython.display import HTML
parallel = px.parallel_coordinates(table, dimensions=['top_n', 'volatility_60d_max', 'annual_return', 'max_drawdown', 'sharpe_ratio'], color='sharpe_ratio')
display(HTML(pio.to_html(parallel, full_html=False, include_plotlyjs=True)))

## 4. 导出

In [ ]:
from datetime import datetime
path = ROOT / 'output/vectorbt/param_scans' / f"notebook_scan_{datetime.now():%Y%m%d_%H%M%S}.parquet"
scan_results.to_parquet(path)